# 기업별 뉴스 수집 노트북

팀원이 기업명을 바꿔 네이버 뉴스 검색 결과와 기사 본문 후보를 수집할 수 있는 노트북입니다. 위에서부터 셀을 차례대로 실행하세요.

- 뉴스 1건당 1행: `document_id`, `title`, `original_url`, `published_at`, `content`
- 이미지는 제외하고 본문을 마크다운으로 변환합니다. 주제·관계 추출은 후속 GraphRAG 단계에서 진행합니다.
- 자동 추출 결과는 후보입니다. 실제 기사와 기업의 관련성, 본문 정확성, 발행일과 이용 조건을 확인해야 합니다. API의 `pubDate`는 네이버에 제공된 시각입니다.
- 기사 본문 및 검색 결과의 저장·공유·AI 활용 조건은 원문 제공처와 네이버의 이용 조건을 확인하세요.

## 1. 필요한 도구와 API 키

프로젝트 루트의 `.env`에 `NAVER_CLIENT_ID`, `NAVER_CLIENT_SECRET`을 설정하세요. 키를 노트북이나 Git에 넣지 않습니다. `requests`, `pandas`, `python-dotenv`, `beautifulsoup4`, `markdownify`가 필요합니다.

In [ ]:
import os
import re
import html
import time
import hashlib
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from markdownify import markdownify
from IPython.display import display, Markdown

# 노트북이 프로젝트 루트 또는 그 하위에서 실행된다고 가정합니다.
current_path = Path.cwd().resolve()
project_root = next(
    (p for p in [current_path, *current_path.parents] if (p / ".env").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("프로젝트 폴더의 .env를 찾지 못했습니다. 현재 실행 위치를 확인해 주세요.")

load_dotenv(project_root / ".env", override=True)
naver_client_id = os.getenv("NAVER_CLIENT_ID")
naver_client_secret = os.getenv("NAVER_CLIENT_SECRET")
if not naver_client_id or not naver_client_secret:
    raise ValueError(".env에 NAVER_CLIENT_ID와 NAVER_CLIENT_SECRET을 설정해 주세요.")

headers = {
    "X-NCP-APIGW-API-KEY-ID": naver_client_id,
    "X-NCP-APIGW-API-KEY": naver_client_secret,
}
print("프로젝트:", project_root)
print("네이버 API 키를 확인했습니다.")

## 2. 수집 대상과 설정

`COMPANY_NAME`을 조사할 기업명으로 바꾸세요. 필요하면 검색어를 직접 추가할 수 있습니다. 현재 결과 스키마에는 기업 ID 컬럼이 없으므로, 여러 기업의 결과는 파일을 구분해 저장하세요.

네이버 뉴스 API는 날짜 범위를 직접 지정할 수 없습니다. 키워드와 정렬에 따라 최대 1,000개 검색 결과만 접근할 수 있어 전체 기사가 보장되지 않습니다. 연도가 검색어에 포함됐다고 그 연도의 기사만 반환되는 것은 아닙니다.

In [ ]:
# 팀원이 수정할 항목
COMPANY_NAME = "아모레퍼시픽"  # 조사할 기업명
START_YEAR = 2023
PAGES_PER_QUERY = 3  # 검색어당 100건씩 최대 10페이지
REQUEST_DELAY = 1   # 기사 요청 사이 대기 시간(초)
BATCH_SIZE = 20     # 한 번에 본문을 가져올 기사 수

if not COMPANY_NAME.strip():
    raise ValueError("COMPANY_NAME에 기업명을 입력해 주세요.")
if not 1 <= PAGES_PER_QUERY <= 10:
    raise ValueError("PAGES_PER_QUERY는 1에서 10 사이로 입력해 주세요.")
if BATCH_SIZE < 1 or REQUEST_DELAY < 0:
    raise ValueError("BATCH_SIZE는 1 이상, REQUEST_DELAY는 0 이상이어야 합니다.")

current_year = pd.Timestamp.now(tz="Asia/Seoul").year
search_queries = [(COMPANY_NAME, "date")]
search_queries += [(f"{COMPANY_NAME} {year}", "sim") for year in range(START_YEAR, current_year + 1)]
# 더 찾고 싶은 검색어가 있다면 아래처럼 추가하세요.
# search_queries.append((f"{COMPANY_NAME} 해외 사업", "sim"))

print("검색어:", search_queries)

## 3. 뉴스 검색과 중복 정리

기사 제목과 링크를 먼저 모읍니다. 검색 결과의 시각을 기준으로 2023년부터 현재까지 필터링하며, 날짜가 없거나 파싱되지 않은 결과는 제외합니다. 저장되는 `published_at`은 네이버 제공 시각을 우선 사용하므로 실제 발행일과 다를 수 있습니다.

In [ ]:
api_url = "https://naverapihub.apigw.ntruss.com/search/v1/news"

def clean_news_text(value):
    return html.unescape(re.sub(r"<[^>]+>", "", value or "")).strip()

search_rows = []
search_errors = []

for query, sort in search_queries:
    print("검색:", query, "· 정렬:", sort)
    for page in range(PAGES_PER_QUERY):
        start = 1 + page * 100
        try:
            response = requests.get(
                api_url,
                headers=headers,
                params={"query": query, "display": 100, "start": start, "sort": sort},
                timeout=30,
            )
            response.raise_for_status()
            data = response.json()
            items = data.get("items", [])
        except (requests.RequestException, ValueError) as error:
            search_errors.append({"query": query, "start": start, "error": type(error).__name__})
            print("  요청 실패:", type(error).__name__)
            break

        for item in items:
            search_rows.append({
                "title": clean_news_text(item.get("title")),
                "original_url": item.get("originallink"),
                "naver_provided_at": item.get("pubDate"),
                "query": query,
            })
        print(f"  {start}번째부터 {len(items)}건")
        if len(items) < 100:
            break
        time.sleep(0.3)

columns = ["title", "original_url", "naver_provided_at", "query"]
search_df = pd.DataFrame(search_rows, columns=columns)
search_df["naver_provided_at"] = pd.to_datetime(
    search_df["naver_provided_at"], errors="coerce", utc=True
).dt.tz_convert("Asia/Seoul")
start_date = pd.Timestamp(year=START_YEAR, month=1, day=1, tz="Asia/Seoul")
end_date = pd.Timestamp.now(tz="Asia/Seoul")
valid_url = search_df["original_url"].fillna("").str.match(r"^https?://")
news_candidates_df = (
    search_df[valid_url & search_df["naver_provided_at"].between(start_date, end_date)]
    .drop_duplicates(subset=["original_url"])
    .reset_index(drop=True)
)
print(f"기간·주소 중복 정리 후: {len(news_candidates_df)}건")
if search_errors:
    print("일부 검색 요청이 실패했습니다. 아래 결과를 확인하세요.")
    display(pd.DataFrame(search_errors))
display(news_candidates_df.head(10))

In [ ]:
# 검색 결과의 연도별 분포 (없으면 0건)
year_counts = news_candidates_df["naver_provided_at"].dt.year.value_counts().sort_index()
print("연도별 기사 후보")
display(year_counts)
print("전체 기사 후보:", len(news_candidates_df), "건")

## 4. 기사 본문 추출 함수

확인한 언론사 선택자를 우선 사용하고, 본문을 찾지 못하면 공통 선택자도 시도합니다. 공통 선택자가 잡은 영역에는 관련 기사나 광고가 섞일 수 있습니다. 사이트 구조 변경, JavaScript, 접근 제한, 인코딩 때문에 실패할 수 있습니다. 스카이데일리 PC 주소·인코딩은 기존에 확인한 기사 한 건에만 적용합니다.

In [ ]:
# 지금까지 확인한 사이트별 본문 위치 (추가 사례 확인 후 변경 가능)
site_selectors = {
    "www.asiatime.co.kr": ["#articleContent"],
    "www.metroseoul.co.kr": [".article-txt-contents"],
    "biz.chosun.com": [".article-body"],
    "www.etnews.com": ["#articleBody"],
    "www.techholic.co.kr": ["#articleBody"],
    "www.jejudomin.co.kr": ["#article-view-content-div"],
    "www.newsian.co.kr": ["#article-view-content-div"],
    "www.pinpointnews.co.kr": ["#article-view-content-div"],
    "www.newsis.com": ["article"],
}
common_selectors = [
    "#article-view-content-div", "#articleContent", ".article-txt-contents",
    "#article-view-content", "#articleBody", "#article-body",
    "#newsContent", "#newsct_article", ".article-body", ".article-content",
    ".news_view", "article",
]


def get_article_markdown(url):
    result = {"original_url": url, "requested_url": url, "selector": None,
              "markdown": None, "error": None}
    try:
        domain = urlparse(url).netloc.lower()
        # 확인된 스카이데일리 사진 기사만 PC 주소와 인코딩 사용
        sky_example = domain == "m.skyedaily.com" and "ID=314172" in url
        if sky_example:
            result["requested_url"] = "https://www.skyedaily.com/news/news_view.html?ID=314172"

        response = requests.get(result["requested_url"], timeout=30)
        response.raise_for_status()
        page = response.content.decode("cp949") if sky_example else response.content
        soup = BeautifulSoup(page, "html.parser")
        selectors = (["#gisaview"] if sky_example else site_selectors.get(domain, [])) + common_selectors

        article_tag = None
        for selector in dict.fromkeys(selectors):
            candidate = soup.select_one(selector)
            if candidate and candidate.get_text(" ", strip=True):
                article_tag = candidate
                result["selector"] = selector
                break
        if article_tag is None:
            result["error"] = "본문 영역을 찾지 못했습니다."
            return result

        # 이미지·스크립트를 변환 전에 제거합니다. 사진 설명 문장은 남을 수 있습니다.
        for tag in article_tag.select("img, script, style, iframe"):
            tag.decompose()
        content = markdownify(str(article_tag), heading_style="ATX", table_infer_header=True).strip()
        if not content:
            result["error"] = "이미지 제거 후 본문이 비어 있습니다."
            return result
        result["markdown"] = content
    except requests.RequestException as error:
        result["error"] = f"페이지 요청 실패: {type(error).__name__}"
    except UnicodeDecodeError:
        result["error"] = "문자 인코딩 확인 필요"
    except Exception as error:
        result["error"] = f"변환 실패: {type(error).__name__}"
    return result

## 5. 본문을 20건씩 수집

먼저 초기화 셀을 **한 번만** 실행하세요. 이후 수집 셀을 다시 실행할 때마다 다음 묶음으로 이동합니다. 실패한 기사는 최종 표에서 제외하고 실패 목록에 기록합니다. 노트북을 다시 시작하면 메모리의 수집 결과가 사라집니다.

In [ ]:
# 초기화: 한 번만 실행 (다시 실행하면 메모리의 수집 기록이 초기화됩니다)
news_records_df = pd.DataFrame(columns=[
    "document_id", "title", "original_url", "published_at", "content"
])
attempted_urls = set()
collection_errors = []
print("수집 준비 완료")

In [ ]:
# 다음 기사 묶음 처리: 추가 수집 시 이 셀만 다시 실행
remaining = news_candidates_df[
    ~news_candidates_df["original_url"].isin(attempted_urls)
]
batch = remaining.head(BATCH_SIZE)
print(f"남은 후보 {len(remaining)}건 · 이번 처리 {len(batch)}건")

for number, (_, row) in enumerate(batch.iterrows(), 1):
    article_url = row["original_url"]
    attempted_urls.add(article_url)
    result = get_article_markdown(article_url)
    if result["error"]:
        collection_errors.append({"title": row["title"], "url": article_url, "error": result["error"]})
        print(f"[{number}/{len(batch)}] 제외 · {row['title']}")
    else:
        url_hash = hashlib.sha256(article_url.encode("utf-8")).hexdigest()[:16]
        record = {
            "document_id": f"NEWS-{url_hash}",
            "title": row["title"],
            "original_url": article_url,
            "published_at": row["naver_provided_at"].isoformat(),
            "content": result["markdown"],
        }
        news_records_df.loc[len(news_records_df)] = record
        print(f"[{number}/{len(batch)}] 본문 후보 · {row['title']}")
    time.sleep(REQUEST_DELAY)

print(f"본문 후보 {len(news_records_df)}건 · 실패 {len(collection_errors)}건")

## 6. 결과 확인

본문을 찾았다는 뜻이지 정확성과 기업 관련성이 검증됐다는 뜻은 아닙니다. 기사 0번을 화면에서 확인하고, 필요하면 행 번호를 바꾸세요.

In [ ]:
with pd.option_context("display.max_colwidth", 120, "display.max_columns", None):
    display(news_records_df)
if collection_errors:
    display(pd.DataFrame(collection_errors))
if not news_records_df.empty:
    print("첫 번째 기사:", news_records_df.iloc[0]["title"])
    display(Markdown(news_records_df.iloc[0]["content"]))

## 7. JSONL 저장 (조건 확인 후 실행)

공용 collected 폴더에 저장합니다. 기존 파일은 덮어쓰지 않습니다. 기사 본문의 저장·공유·AI 활용 조건이 확인된 자료만 저장하세요. 아래 예시는 현재 표 전체를 저장하므로, 필요하다면 먼저 `news_records_df`를 검토 대상에 맞게 정리해야 합니다.

In [ ]:
# 저장을 원할 때에만 실행하세요.
output_path = project_root / "data" / "collected" / "news.jsonl"

if news_records_df.empty:
    raise ValueError("저장할 기사 데이터가 없습니다.")
if output_path.exists():
    raise FileExistsError(f"기존 파일이 있습니다. 다른 파일명을 사용하세요: {output_path}")

output_path.parent.mkdir(parents=True, exist_ok=True)
# x 모드: 파일이 이미 존재하면 쓰지 않습니다.
with output_path.open("x", encoding="utf-8") as file:
    file.write(news_records_df.to_json(orient="records", lines=True, force_ascii=False))
print(f"저장 완료: {output_path} · {len(news_records_df)}건")